# Diamond Price Prediction

This session predicts the price of a diamond based on its size.
The idea is simple: a bigger diamond usually costs more. We will use a real dataset
to see this pattern and train a model to predict prices.

Instructions for running this notebook:
- Run each cell in order using Shift + Enter.
- Cells marked LIVE CODING are meant to be written together with the audience during the session.

## 1. Load the Dataset

**About this dataset:** this is a real dataset of diamond sales. Each row is one diamond.
The main columns we will use are `carat` (the weight/size of the diamond) and `price`
(its price in dollars). We will not use the other columns in our model, to keep it simple,
but here is what they mean:

- `cut`: quality of the cut, from worst to best: Fair, Good, Very Good, Premium, Ideal
- `color`: diamond color grade, from D (best, colorless) to J (worst, slightly yellow)
- `clarity`: how flawless the diamond is, from worst to best: I1, SI2, SI1, VS2, VS1, VVS2, VVS1, IF
- `depth`: total depth as a percentage, roughly z divided by the average of x and y
- `table`: width of the flat top of the diamond, relative to its widest point
- `x`: length of the diamond in millimeters
- `y`: width of the diamond in millimeters
- `z`: depth (height) of the diamond in millimeters

In [ ]:
import seaborn as sns

# Load a real dataset of diamonds
diamonds = sns.load_dataset("diamonds")
diamonds.head()


In [ ]:
# Basic info about the dataset
print("Rows and columns:", diamonds.shape)
print()
print(diamonds.info())


This dataset has over 50,000 rows. For a simple, easy-to-follow session, we will
work with a smaller sample of 300 diamonds.

In [ ]:
# Take a random sample of 300 diamonds to keep things simple
df = diamonds.sample(300, random_state=42).reset_index(drop=True)
print("Sample size:", df.shape)
df.head()


## 2. Simple Data Cleaning

Before using data, always check for missing values.

In [ ]:
# Check for missing values in each column
df.isnull().sum()


There are no missing values in this dataset, so no cleaning is needed here.
In real projects, this check is always done first, since missing values can break a model.

## 3. Simple Visualization

Let's look at the relationship between `carat` (size) and `price`.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["carat"], df["price"])
plt.title("Diamond Size vs Price")
plt.xlabel("Carat (size)")
plt.ylabel("Price (dollars)")
plt.show()


### LIVE CODING 1

Draw a scatter plot showing the relationship between `depth` and `price`.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df[...], df[...])
plt.title("Diamond Depth vs Price")
plt.xlabel("Depth")
plt.ylabel("Price (dollars)")
plt.show()


## 4. Building a Prediction Model

We will train a simple Linear Regression model to predict `price` from `carat` only.
One input, one output, easy to follow.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Input: carat (size)      Output: price
X = df[["carat"]]
y = df["price"]

# Split into training data and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained.")


In [ ]:
# Check how well the model performs on data it has not seen before
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE (Mean Absolute Error):", round(mae, 2))
print("RMSE (Root Mean Squared Error):", round(rmse, 2))
print("R-squared:", round(r2, 3))


**What these three numbers mean, in simple terms:**

- **MAE** - the average size of the mistake, in dollars. Treats every mistake equally.
- **RMSE** - also the average mistake, in dollars, but it punishes big mistakes much more
  than small ones. Diamond prices jump quickly at the high end, so a few very expensive
  diamonds can cause large misses. RMSE will always be equal to or higher than MAE, and
  the bigger the gap between them, the more the model is struggling with a few large errors
  rather than being consistently a little bit off.
- **R-squared** - how much of the price pattern the model explains, on a scale from 0 to 1.
  1.0 would mean the model predicts price perfectly. 0 would mean it is no better than
  always guessing the average price. Closer to 1 is better.

In [ ]:
import matplotlib.pyplot as plt

# Plot the actual data points and the line the model learned
plt.scatter(df["carat"], df["price"], label="Actual data")
plt.plot(X_test, predictions, color="red", label="Model prediction")
plt.title("Linear Regression: Carat vs Price")
plt.xlabel("Carat (size)")
plt.ylabel("Price (dollars)")
plt.legend()
plt.show()


In [ ]:
# Predict the price for a new diamond size
new_carat = [[1.0]]
predicted_price = model.predict(new_carat)

print("Predicted price for a 1.0 carat diamond:", round(predicted_price[0], 2))


### LIVE CODING 2

Predict the price for a diamond size of your choice using the trained model.

In [ ]:
# Step 1: choose a carat value
new_carat = [[...]]

# Step 2: predict the price
predicted_price = model.predict(...)

print("Predicted price:", predicted_price[0])


## 5. Improving the Model

Our model only looks at `carat`. But two diamonds of the same size can have very
different prices, depending on their quality. Let's add `cut`, `color`, and `clarity`
to the model and see if the predictions improve.

In [ ]:
# cut, color, and clarity are text values ("Ideal", "Fair", etc.)
# A model only understands numbers, so we convert each quality level into a number.
# Lower number = lower quality, higher number = higher quality.

cut_quality = {"Fair": 0, "Good": 1, "Very Good": 2, "Premium": 3, "Ideal": 4}
color_quality = {"J": 0, "I": 1, "H": 2, "G": 3, "F": 4, "E": 5, "D": 6}
clarity_quality = {"I1": 0, "SI2": 1, "SI1": 2, "VS2": 3, "VS1": 4, "VVS2": 5, "VVS1": 6, "IF": 7}

df["cut_score"] = df["cut"].map(cut_quality)
df["color_score"] = df["color"].map(color_quality)
df["clarity_score"] = df["clarity"].map(clarity_quality)

df[["carat", "cut_score", "color_score", "clarity_score", "price"]].head()


In [ ]:
# Train a new model using carat AND the three quality scores
X_improved = df[["carat", "cut_score", "color_score", "clarity_score"]]
y_improved = df["price"]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_improved, y_improved, test_size=0.2, random_state=42)

improved_model = LinearRegression()
improved_model.fit(X_train2, y_train2)

print("Improved model trained.")


In [ ]:
# Compare the improved model to our first model
improved_predictions = improved_model.predict(X_test2)

mae_improved = mean_absolute_error(y_test2, improved_predictions)
rmse_improved = np.sqrt(mean_squared_error(y_test2, improved_predictions))
r2_improved = r2_score(y_test2, improved_predictions)

print("Original model (carat only)      MAE:", round(mae, 2), " RMSE:", round(rmse, 2), " R-squared:", round(r2, 3))
print("Improved model (carat + quality)  MAE:", round(mae_improved, 2), " RMSE:", round(rmse_improved, 2), " R-squared:", round(r2_improved, 3))


**What happened:** adding the quality of the diamond, not just its size, gave the model
more useful information to work with. This is a simple example of a common idea in
machine learning: a model is often only as good as the information it is given.

## Summary

Covered in this session:
- Loading a real dataset
- Checking for missing values
- Visualizing the relationship between two values
- Training a simple Linear Regression model
- Measuring prediction error with MAE, RMSE, and R-squared
- Using the model to predict a new value
- Improving the model by adding more relevant information (cut, color, clarity)

Reference: https://scikit-learn.org/stable/modules/linear_model.html